In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!rm -rf Cap3_MedicionPobreza
!git clone https://github.com/LizamaD/Cap3_MedicionPobreza.git

Cloning into 'Cap3_MedicionPobreza'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 193 (delta 110), reused 152 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 4.40 MiB | 50.08 MiB/s, done.
Resolving deltas: 100% (110/110), done.


In [3]:
import pandas as pd

path_bases = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"
path_pobreza = '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Base final/pobreza_24.csv'


# --- Carga tus datos crudos ---
poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")
trabajos_raw = pd.read_csv(f"{path_bases}trabajos.csv")
gastospersona_raw = pd.read_csv(f"{path_bases}gastospersona.csv")
gastoshogar_raw = pd.read_csv(f"{path_bases}gastoshogar.csv")
ingresos_raw = pd.read_csv(f"{path_bases}ingresos.csv")
pobreza_raw = pd.read_csv(path_pobreza)

/tmp/ipykernel_12658/1323284465.py:8: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
/tmp/ipykernel_12658/1323284465.py:9: DtypeWarning: Columns (1,4,27,44) have mixed types. Specify dtype option on import or set low_memory=False.
  viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
/tmp/ipykernel_12658/1323284465.py:10: DtypeWarning: Columns (45,49,53,61,63,69,75,77,81,83,85) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")


In [4]:
%cd /content/Cap3_MedicionPobreza

/content/Cap3_MedicionPobreza


In [5]:
from src.trabajos import process_trabajos
from src.ingresos import generar_ingreso_deflactado_ago2024
from src.gastoshogar import procesar_gastos_enigh
from src.gastospersona import procesar_gastos_persona_enigh
from src.hogares import process_hogares
from src.poblacion import process_poblacion
from src.viviendas import process_viviendas
from src.pipeline import create_master_table, impute_data, prepare_and_split_for_autoencoder

In [6]:
# --- Crea la tabla maestra ---
# Esta función procesa, une todo y guarda el resultado en "master_table.csv"
master_df = create_master_table(
    pob_keys=pobreza_raw,
    pob_df=poblacion_raw,
    viv_df=viviendas_raw,
    hog_df=hogares_raw,
    trab_df=trabajos_raw,
    gasper_df=gastospersona_raw,
    gashog_df=gastoshogar_raw,
    ing_df=ingresos_raw,
    output_path=f"{path_bases}enigh_master_table.csv" # Ruta sugerida
)

# --- Imputa los datos faltantes ---
# Esta función toma la tabla maestra y la deja lista para el modelado
df_final_limpio = impute_data(master_df)

# --- Verifica el resultado ---
print("\nDataFrame final después de la imputación:")
print(df_final_limpio.head())

# Confirma que no hay valores nulos
print("\nSuma de nulos por columna en el DF final:")
print(df_final_limpio.isnull().sum().sum()) # Debería ser 0


Procesando tablas individuales...


/content/Cap3_MedicionPobreza/src/ingresos.py:34: FutureWarning: The provided callable <function sum at 0x7d76670a0360> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  aguinaldo = pd.pivot_table(


Uniendo tablas en una tabla maestra...
Tabla maestra creada con 308444 filas y 160 columnas.
Guardando tabla maestra en '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/enigh_master_table.csv'...
Iniciando proceso de imputación de datos...
  - Categórica 'tam_emp': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact': Imputando NaNs con '__MISSING__'.
  - Categórica 'socios': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_nr1': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_nr2': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_resp': Imputando NaNs con '__MISSING__'.
  - Categórica 'otra_act': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact2': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact3': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact4': Imputando NaNs con '__MISSING__'.
  - Categórica 'lugar': Imputando NaNs con '__MISSING__'.
  - Categórica 'conf_pers': Imputando NaNs con '__MISSING__'.

/content/Cap3_MedicionPobreza/src/pipeline.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[flag_col_name] = df_clean[col].isnull().astype(int)
/content/Cap3_MedicionPobreza/src/pipeline.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[flag_col_name] = df_clean[col].isnull().astype(int)
/content/Cap3_MedicionPobreza/src/pipeline.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

0


In [7]:
# Define tu ruta de salida en Google Drive
output_directory = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"

# Llama a la nueva función para procesar y guardar todo
df_processed, df_pobres, df_no_pobres = prepare_and_split_for_autoencoder(
    df_final_limpio, 
    output_directory
)

# Ahora ya tienes tus DataFrames listos para la estrategia de descubrimiento:
# - df_no_pobres: Para entrenar y validar tu autoencoder.
# - df_pobres: Para pasarlo por el modelo ya entrenado y analizar el error de reconstrucción.



Iniciando preparación y split para el autoencoder...
Convirtiendo 17 columnas categóricas a formato dummy...
El DataFrame ahora tiene 328 columnas después del one-hot encoding.

Aplicando filtro de baja varianza...
Dimensión original de features: 322
Variables eliminadas por baja varianza: 106
Variables que quedaron: 216

Columnas eliminadas:
   ['carencia_pared', 'carencia_techo', 'carencia_luz', 'alim17_1', 'alim17_10']
   ['trapais', 'flag_pres_16', 'flag_pres_19', 'flag_medtrab_3', 'flag_medtrab_4']
   ['flag_medtrab_5', 'flag_medtrab_6', 'no_ing', 'tiene_suel', 'ing_mon_imputed']
   ['ing_lab_imputed', 'ing_ren_imputed', 'ing_tra_imputed', 'parentesco_202', 'parentesco_205']
   ['parentesco_302', 'parentesco_303', 'parentesco_304', 'parentesco_501', 'parentesco_601']
   ['parentesco_602', 'parentesco_604', 'parentesco_605', 'parentesco_606', 'parentesco_607']
   ['parentesco_608', 'parentesco_610', 'parentesco_611', 'parentesco_612', 'parentesco_613']
   ['parentesco_614', 'paren

In [12]:
df_pobres.head(2)

,folioviv,foliohog,numren,pobreza,pobreza_e,factor,edad,afrod,hablaind,etnia,...,socios_2,socios___MISSING__,otra_act_1,otra_act_2,otra_act___MISSING__,tipoact2___MISSING__,lugar___MISSING__,conf_pers___MISSING__,inst_00,inst___MISSING__
26,100002505,1,1,1.0,0.0,196,33,0,0,0,...,0,1,0,0,1,1,1,1,0,1
27,100002505,1,2,1.0,0.0,196,34,0,0,0,...,0,1,0,0,1,1,1,1,0,1
